# 6) Redispatch and N-1 security

## Objectives

At the end of this notebook you should be able to:

- state the **N-1 criterion** and explain why a dispatch that respects every line limit can still be insecure
- derive **line outage distribution factors** (LODFs) from the PTDFs, and check them against recomputing the network without the line
- run a **contingency analysis**: every single line outage, every line's flow afterwards
- write a **security-constrained** dispatch LP (preventive SCOPF), reproduce it with PyPSA's `optimize_security_constrained`, and read its nodal prices
- price the **cost of security** and the **redispatch** a market dispatch needs to become N-1 secure

Lesson 4 found the cheapest dispatch that keeps every line within its limit — with every line in service. Real operators plan for failure: European rules require transmission system operators to keep the system secure after the loss of any single element. This lesson asks what that costs on lesson 4's network.

In [1]:
import logging
import warnings

import linopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

warnings.filterwarnings("ignore")
for name in ("pypsa", "linopy"):
    logging.getLogger(name).setLevel(logging.ERROR)

### Lesson 4's network and its dispatch

In [2]:
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
LINES.index = [f"{a}-{b}" for a, b in zip(LINES.bus0, LINES.bus1)]
X_KM, LIMIT = 0.32, 400.0                      # the same 400 MW limit before and after an outage
LOAD = pd.Series({"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250, "Velebit": 0})[BUSES]
GENS = pd.DataFrame([("wind Velebit", "Velebit", 600, 0.0), ("hydro Split", "Split", 700, 30.0),
                     ("gas Rijeka", "Rijeka", 600, 90.0), ("coal Osijek", "Osijek", 400, 80.0),
                     ("import Zagreb", "Zagreb", 800, 130.0)],
                    columns=["name", "bus", "p_max_MW", "cost_EUR_per_MWh"]).set_index("name")


def ptdf(lines=LINES, reference="Zagreb"):
    idx = {b: i for i, b in enumerate(BUSES)}
    b = 1 / (X_KM * lines.km.to_numpy() / (400 ** 2 / 100))
    A = np.zeros((len(lines), len(BUSES)))
    A[range(len(lines)), lines.bus0.map(idx)] = 1
    A[range(len(lines)), lines.bus1.map(idx)] = -1
    B = A.T @ np.diag(b) @ A
    keep = [i for i, name in enumerate(BUSES) if name != reference]
    B_inv = np.zeros_like(B)
    B_inv[np.ix_(keep, keep)] = np.linalg.inv(B[np.ix_(keep, keep)])
    return pd.DataFrame(np.diag(b) @ A @ B_inv, index=lines.index, columns=BUSES)


PTDF = ptdf()
G = np.zeros((len(BUSES), len(GENS)))
for k, bus in enumerate(GENS.bus):
    G[BUSES.index(bus), k] = 1
GI = pd.Index(GENS.index, name="gen")
COST = xr.DataArray(GENS.cost_EUR_per_MWh.to_numpy(), coords=[GI])


def flows(dispatch, ptdf_matrix=PTDF):
    return pd.Series(ptdf_matrix.to_numpy() @ (G @ dispatch[GENS.index].to_numpy() - LOAD.to_numpy()), index=ptdf_matrix.index)

### Line outage distribution factors

When line k fails, the power it carried must find other paths. In the DC model the redistribution is linear: the flow on line ℓ after the outage is

f_ℓ after = f_ℓ + LODF_ℓk · f_k

The LODF follows from the PTDFs. Losing line k (from bus i to bus j) is equivalent to a transfer from i to j that exactly cancels the line's own flow; the PTDF of that transfer on line ℓ is PTDF_ℓi − PTDF_ℓj, and the transfer has to be scaled up because part of it would flow over line k itself:

**LODF_ℓk = (PTDF_ℓi − PTDF_ℓj) / (1 − (PTDF_ki − PTDF_kj))**, and LODF_kk = −1.

In [3]:
def lodf():
    """LODF[l, k]: change of the flow on line l per MW that line k carried before it failed."""
    out = pd.DataFrame(0.0, index=LINES.index, columns=LINES.index)
    for k in LINES.index:
        transfer = PTDF[LINES.bus0[k]] - PTDF[LINES.bus1[k]]       # flows per MW moved from bus0 to bus1 of line k
        out[k] = transfer / (1 - transfer[k])
        out.loc[k, k] = -1.0
    return out


LODF = lodf()
LODF.round(3)

                Zagreb-Rijeka  Rijeka-Velebit  Velebit-Split  Zagreb-Split  \
Zagreb-Rijeka          -1.000          -1.000         -1.000         0.592   
Rijeka-Velebit         -1.000          -1.000         -1.000         0.592   
Velebit-Split          -1.000          -1.000         -1.000         0.592   
Zagreb-Split            0.644           0.644          0.644        -1.000   
Zagreb-Osijek           0.356           0.356          0.356         0.408   
Osijek-Split            0.356           0.356          0.356         0.408   

                Zagreb-Osijek  Osijek-Split  
Zagreb-Rijeka           0.444         0.444  
Rijeka-Velebit          0.444         0.444  
Velebit-Split           0.444         0.444  
Zagreb-Split            0.556         0.556  
Zagreb-Osijek          -1.000        -1.000  
Osijek-Split           -1.000        -1.000  

In [4]:
def nodal(security=False, post_limit=LIMIT, contingencies=None):
    """Lesson 4's dispatch; with security=True also every line within post_limit after each single line outage."""
    contingencies = list(LINES.index) if contingencies is None else contingencies
    m = linopy.Model()
    p = m.add_variables(lower=0, upper=xr.DataArray(GENS.p_max_MW.to_numpy(), coords=[GI]), name="p")
    m.add_constraints(p.sum() == LOAD.sum(), name="balance")
    M, c0 = PTDF.to_numpy() @ G, PTDF.to_numpy() @ LOAD.to_numpy()
    f = {l: (p * xr.DataArray(M[i], coords=[GI])).sum() - float(c0[i]) for i, l in enumerate(LINES.index)}
    sens = {}                                                         # sensitivity of each constraint to bus injections
    for l in LINES.index:
        m.add_constraints(f[l] <= LIMIT, name=f"upper {l}")
        m.add_constraints(f[l] >= -LIMIT, name=f"lower {l}")
        sens[f"upper {l}"] = sens[f"lower {l}"] = PTDF.loc[l]
    if security:
        for k in contingencies:
            for l in LINES.index.drop(k):
                post = f[l] + LODF.loc[l, k] * f[k]
                m.add_constraints(post <= post_limit, name=f"upper {l} | {k} out")
                m.add_constraints(post >= -post_limit, name=f"lower {l} | {k} out")
                sens[f"upper {l} | {k} out"] = sens[f"lower {l} | {k} out"] = PTDF.loc[l] + LODF.loc[l, k] * PTDF.loc[k]
    m.add_objective((p * COST).sum())
    m.solve(solver_name="highs", output_flag=False)
    lmp = pd.Series(float(m.constraints["balance"].dual), index=BUSES)
    binding = {}
    for name, s in sens.items():
        mu = float(m.constraints[name].dual)
        if abs(mu) > 1e-9:
            lmp = lmp + s * mu
            binding[name] = mu
    return {"cost": float(m.objective.value), "dispatch": p.solution.to_pandas(), "lmp": lmp, "binding": binding}


N0 = nodal()
f0 = flows(N0["dispatch"])
worst = 0.0
for k in LINES.index:                                                 # LODF prediction against recomputing without line k
    without_k = LINES.drop(k)
    recomputed = flows(N0["dispatch"], ptdf(without_k))
    predicted = f0.drop(k) + LODF.loc[without_k.index, k] * f0[k]
    worst = max(worst, (predicted - recomputed).abs().max())
print(f"LODF prediction = recomputed flows without the line, to {worst:.1e} MW, for all six outages")

LODF prediction = recomputed flows without the line, to 1.1e-12 MW, for all six outages


The LODFs reproduce the network without each line to 10⁻¹² MW — in the DC model they are exact, not an approximation. The matrix also shows the network's structure. Rijeka and Velebit each have only two lines, so Zagreb–Rijeka, Rijeka–Velebit and Velebit–Split form one **chain**: if any link fails, the whole chain stops carrying power (an LODF of −1 for each other's outages), and what it carried must take the other routes — 64.4 % over Zagreb–Split and 35.6 % through Osijek. Osijek is the same: Zagreb–Osijek and Osijek–Split fail together. Seen from the grid, this network has three parallel corridors between Zagreb and the south, not six independent lines — and the loss of one corridor shifts its whole flow onto the other two.

### Contingency analysis of lesson 4's dispatch

Lesson 4's dispatch keeps every line at or below 400 MW with all lines in service — **N-0 secure**. Now take each line out in turn:

In [5]:
def contingency_table(dispatch):
    """For each single line outage: the most loaded remaining line and its flow (MW)."""
    f = flows(dispatch)
    rows = {}
    for k in LINES.index:
        after = (f + LODF[k] * f[k]).drop(k)
        rows[f"{k} out"] = {"worst line": after.abs().idxmax(), "flow_MW": after.abs().max(),
                            "overloaded lines": int((after.abs() > LIMIT + 1e-6).sum())}
    return pd.DataFrame(rows).T


print(f"N-0 dispatch: {N0['cost']:,.1f} EUR/h, largest flow {f0.abs().max():.1f} MW")
contingency_table(N0["dispatch"])

N-0 dispatch: 81,956.3 EUR/h, largest flow 400.0 MW


                       worst line     flow_MW overloaded lines
Zagreb-Rijeka out    Zagreb-Split   515.30205                2
Rijeka-Velebit out   Zagreb-Split   515.30205                2
Velebit-Split out   Zagreb-Rijeka  484.951456                2
Zagreb-Split out    Zagreb-Rijeka  552.412324                2
Zagreb-Osijek out   Zagreb-Rijeka  501.078749                2
Osijek-Split out    Zagreb-Rijeka  434.412082                2

Lesson 4's dispatch is secure only as long as nothing fails. **Every** single line outage overloads two lines — always a whole corridor — and the worst case, losing Zagreb–Split, puts 552 MW on Zagreb–Rijeka. The LODF shows how: Zagreb–Split carried 258 MW from Split to Zagreb; 59.2 % of it re-routes through the Velebit–Rijeka corridor, which was already at its 400 MW limit — 400 + 0.592 × 258 = 552 MW. A line that is not connected to Rijeka at all decides how much Rijeka's lines must carry. This is why operators do not stop at N-0: a dispatch that sits exactly on the limits, as an optimal dispatch always does, leaves no room for any failure.

### Preventive security-constrained dispatch

Add the post-outage flows to the LP: for every outage k and every other line ℓ, f_ℓ + LODF_ℓk · f_k must stay within the limit — 6 × 5 × 2 = 60 extra constraints. The dispatch is chosen **before** anything fails, so that whatever fails, no line is overloaded: **preventive** security. PyPSA does the same with `optimize_security_constrained`.

In [6]:
N1 = nodal(security=True)

n = pypsa.Network()
n.add("Bus", BUSES, v_nom=400)
for bus, load in LOAD.items():
    if load:
        n.add("Load", f"{bus} load", bus=bus, p_set=load)
for name, g in GENS.iterrows():
    n.add("Generator", name, bus=g.bus, p_nom=g.p_max_MW, marginal_cost=g.cost_EUR_per_MWh)
for name, line in LINES.iterrows():
    n.add("Line", name, bus0=line.bus0, bus1=line.bus1, x=X_KM * line.km, r=0.0, s_nom=LIMIT)
n.optimize.optimize_security_constrained(branch_outages=n.lines.index, solver_name="highs",
                                         log_to_console=False, progress=False)
assert abs(n.objective - N1["cost"]) < 1e-6 * N1["cost"]
assert np.abs(n.generators_t.p.iloc[0][GENS.index].to_numpy() - N1["dispatch"].to_numpy()).max() < 1e-4
assert np.abs(n.buses_t.marginal_price.iloc[0][BUSES].to_numpy() - N1["lmp"].to_numpy()).max() < 1e-6
print(f"own SCOPF = PyPSA: cost {N1['cost']:,.1f} EUR/h, dispatch and every nodal price agree")
print("binding constraints (shadow price, EUR/MWh):", {k: round(v, 2) for k, v in N1["binding"].items()})
print("\ndispatch (MW):")
print(pd.DataFrame({"N-0": N0["dispatch"], "N-1": N1["dispatch"]}).round(1).to_string())
print("\nnodal prices (EUR/MWh):")
print(pd.DataFrame({"N-0": N0["lmp"], "N-1": N1["lmp"]}).round(2).to_string())

own SCOPF = PyPSA: cost 107,913.8 EUR/h, dispatch and every nodal price agree
binding constraints (shadow price, EUR/MWh): {'upper Velebit-Split | Rijeka-Velebit out': -2.41, 'lower Zagreb-Rijeka | Zagreb-Split out': 62.41, 'lower Rijeka-Velebit | Zagreb-Split out': 106.55}

dispatch (MW):
                 N-0    N-1
gen                        
wind Velebit   485.0  400.0
hydro Split    700.0  550.9
gas Rijeka     300.0  300.0
coal Osijek    400.0  400.0
import Zagreb   15.0  249.1

nodal prices (EUR/MWh):
            N-0    N-1
Zagreb   130.00  130.0
Rijeka    90.00   90.0
Split     56.80   30.0
Osijek    98.45   86.9
Velebit    0.00    0.0


In [7]:
after_n0 = contingency_table(N0["dispatch"]).flow_MW
after_n1 = contingency_table(N1["dispatch"]).flow_MW
fig, ax = plt.subplots(figsize=(8.5, 3.6))
x = np.arange(len(after_n0))
ax.bar(x - 0.2, after_n0, width=0.4, label="lesson 4's dispatch (N-0 secure)", color="tab:red")
ax.bar(x + 0.2, after_n1, width=0.4, label="security-constrained dispatch (N-1)", color="tab:green")
ax.axhline(LIMIT, color="black", ls="--", lw=1, label="limit")
ax.set_xticks(x, [k.replace(" out", "\nout") for k in after_n0.index], fontsize=8)
ax.set(ylabel="most loaded line after the outage (MW)", title="Every single line outage, before and after securing the dispatch",
       ylim=(0, 720))
ax.legend(fontsize=8, loc="upper center", ncol=3)
plt.tight_layout()
plt.show()

Securing the dispatch against all six outages costs **107,914 €/h — 25,958 €/h (32 %) more** than the N-0 dispatch, and PyPSA's `optimize_security_constrained` gives the same cost, dispatch and nodal prices. The dispatch changes along the southern corridors: wind at Velebit falls from 485 to 400 MW, hydro at Split from 700 to 551 MW, and imports at Zagreb rise from 15 to 249 MW. Power that the network could carry with all lines in service is now held back, because the network must also carry it **after** a loss.

The binding constraints are post-contingency ones: the Velebit–Rijeka corridor after the loss of Zagreb–Split (shadow prices 62.41 and 106.55 €/MWh), and Velebit–Split after the loss of Rijeka–Velebit. They set the prices exactly as lesson 4's congested lines did — through their sensitivities, PTDF + LODF × PTDF. Split's price falls from 56.80 to 30 €/MWh: its hydro plant is now partly dispatched and so becomes the marginal unit there. Osijek's falls from 98.45 to 86.90 €/MWh. After every outage, the most loaded line now stays at or below 400 MW.

### Redispatch to N-1: what a zonal market now costs

In [8]:
one_zone = pd.Series({"wind Velebit": 600.0, "hydro Split": 700.0, "gas Rijeka": 200.0, "coal Osijek": 400.0,
                      "import Zagreb": 0.0})                     # lesson 5's one-zone market dispatch, 71,000 EUR/h
market_cost = float((one_zone * GENS.cost_EUR_per_MWh).sum())
rows = {}
for label, target in {"to N-0 (lesson 5)": N0, "to N-1": N1}.items():
    change = target["dispatch"] - one_zone
    rows[label] = {"redispatch_cost_EUR_per_h": target["cost"] - market_cost,
                   "MW_moved_down": -change[change < 0].sum(), "MW_moved_up": change[change > 0].sum(),
                   "wind_curtailed_MW": -change["wind Velebit"]}
print(f"one-zone market: {market_cost:,.0f} EUR/h")
pd.DataFrame(rows).T.round(1)

one-zone market: 71,000 EUR/h


                   redispatch_cost_EUR_per_h  MW_moved_down  MW_moved_up  \
to N-0 (lesson 5)                    10956.3          115.0        115.0   
to N-1                               36913.8          349.1        349.1   

                   wind_curtailed_MW  
to N-0 (lesson 5)              115.0  
to N-1                         200.0  

From lesson 5's one-zone market dispatch, the operator must now move **349 MW** — 200 MW of wind and 149 MW of hydro down, imports and gas up — at a cost of **36,914 €/h**, three and a half times the 10,956 €/h that N-0 security needed. In a zonal market this is the operator's bill, paid through network tariffs, and it grows with every congested corridor inside the zone. N-1 security is not optional — it is what keeps the lights on when a line trips — but its cost is a strong argument for the measures that reduce it: reinforcing the corridors, rating lines dynamically, and acting **correctively**, after an outage, instead of preventively before it — which works only if a line may carry more for the minutes the operator needs, its short-term emergency rating (the exercise prices it).

## Check your understanding

**Contingencies**

1. Why are the LODFs of Zagreb–Rijeka, Rijeka–Velebit and Velebit–Split all −1 for each other's outages? What does that say about the network's structure?
2. Lesson 4's dispatch loads Zagreb–Rijeka to exactly 400 MW. Why does losing Zagreb–Split push it to 552 MW, although Zagreb–Split is not even connected to Rijeka?

**Security**

1. N-1 security costs 25,958 EUR/h here — 32 % more than the N-0 dispatch. What does society buy with that money?
2. The security-constrained dispatch is chosen before anything fails. What could an operator do instead, if it could act within minutes after an outage?